# RBS Scoring — Kosuri Dataset × Evo2 Log-Probability

Score all 110 RBS sequences from [Kosuri et al. 2013](https://www.nature.com/articles/nbt.2461) using Evo2
conditioned on the real A1_AmtR Cello circuit context, then test whether Evo2 log-probability
correlates with experimentally measured relative translation rate (`mean.xlat`).

**Scoring approach:** For each Kosuri RBS, swap it into the A1_AmtR gate in place of the native
RBS (A1). Build the full-context sequence (`upstream_context + rbs_seq + downstream_context`) and
compute the mean per-token log-probability using the local Evo2 model. This mirrors the
`FULL_CONTEXT` scoring mode used in the ablation study.

**Model:** `evo2_1b_base` for local/Mac; change `MODEL_ID` to `evo2_7b` on a GPU node.

## Section 1 — Install Dependencies

In [ ]:
%pip install -q evo2 pydantic pandas matplotlib seaborn scipy python-dotenv tqdm

## Section 2 — Config

In [ ]:
import os

UCF_PATH          = "../data/raw/Eco1C1G1T1.UCF.json"
GATE_NAME         = "A1_AmtR"
RBS_CSV           = "../data/reference/kosuri_rbs.csv"
MODEL_ID          = "evo2_1b_base"   # evo2_7b on GPU node
OUTPUT_DIR        = "../results/rbs_kosuri_scoring"
STRIP_START_CODON = True             # remove trailing CATATG from Kosuri sequences

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

## Section 3 — Source Code (Inlined)

In [ ]:
# ── PartSpec schema ──────────────────────────────────────────────────────────
from pydantic import BaseModel, Field, field_validator
from typing import Optional, List, Dict, Any
from enum import Enum


class PartType(str, Enum):
    PROMOTER   = "promoter"
    RBS        = "rbs"
    CDS        = "cds"
    TERMINATOR = "terminator"
    OPERATOR   = "operator"
    SPACER     = "spacer"


class HostOrganism(str, Enum):
    ECOLI_K12     = "ecoli_k12"
    ECOLI_BL21    = "ecoli_bl21"
    SACCHAROMYCES = "saccharomyces_cerevisiae"


class PartConstraints(BaseModel):
    min_length:      Optional[int]   = None
    max_length:      Optional[int]   = None
    strength_target: Optional[str]   = None
    inducible_by:    Optional[str]   = None
    repressible_by:  Optional[str]   = None
    gc_content_min:  Optional[float] = None
    gc_content_max:  Optional[float] = None


class PartSpec(BaseModel):
    part_id:            str
    part_type:          PartType
    host:               HostOrganism
    functional_role:    str
    upstream_context:   str
    downstream_context: str
    constraints:        PartConstraints = Field(default_factory=PartConstraints)
    circuit_position:   int
    circuit_total:      int
    reference_seq:      Optional[str] = None
    sbol_component_id:  Optional[str] = None

    @field_validator("upstream_context", "downstream_context", "reference_seq", mode="before")
    @classmethod
    def validate_and_normalize_dna(cls, v):
        if v is None:
            return v
        v = v.upper().strip()
        invalid = set(v) - set("ACGT")
        if invalid:
            raise ValueError(f"Invalid DNA characters: {invalid}")
        return v

    def gc_content(self, sequence: Optional[str] = None) -> float:
        seq = sequence or self.reference_seq or ""
        if not seq:
            return 0.0
        return (seq.count("G") + seq.count("C")) / len(seq)

In [ ]:
# ── Cello UCF parser ─────────────────────────────────────────────────────────
import json
from pathlib import Path

_UCF_TYPE_TO_PART_TYPE: Dict[str, PartType] = {
    "promoter":   PartType.PROMOTER,
    "rbs":        PartType.RBS,
    "cds":        PartType.CDS,
    "terminator": PartType.TERMINATOR,
    "ribozyme":   PartType.SPACER,
    "scar":       PartType.SPACER,
    "operator":   PartType.OPERATOR,
}

_UCF_TYPE_TO_ROLE: Dict[str, str] = {
    "promoter":   "transcription_promoter",
    "rbs":        "ribosome_binding_site",
    "cds":        "coding_sequence",
    "terminator": "transcription_terminator",
    "ribozyme":   "translational_insulator",
    "scar":       "assembly_scar",
    "operator":   "operator_binding_site",
}


def _clean_seq(seq: str) -> str:
    return seq.upper().strip()


def _build_parts_map(ucf_data: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {p["name"]: p for p in ucf_data if p.get("collection") == "parts"}


def _get_cassette_components(structure: Dict[str, Any]) -> List[str]:
    for device in structure.get("devices", []):
        if "cassette" in device["name"]:
            return device["components"]
    return []


def _infer_host(ucf_data: List[Dict[str, Any]]) -> HostOrganism:
    header = next((x for x in ucf_data if x.get("collection") == "header"), {})
    organism = header.get("organism", "").lower()
    if "saccharomyces" in organism:
        return HostOrganism.SACCHAROMYCES
    if "bl21" in organism:
        return HostOrganism.ECOLI_BL21
    return HostOrganism.ECOLI_K12


def _strength_label(ucf_type: Optional[str], params: Dict[str, Any]) -> Optional[str]:
    if ucf_type == "terminator":
        val = params.get("terminator_strength")
        if val is not None:
            return "high" if val > 100 else ("medium" if val > 10 else "low")
    if ucf_type == "ribozyme":
        val = params.get("ribozyme_efficiency")
        if val is not None:
            return "high" if val >= 0.9 else ("medium" if val >= 0.5 else "low")
    return None


def _build_constraints(part: Dict[str, Any]) -> PartConstraints:
    params = {p["name"]: p["value"] for p in part.get("parameters", [])}
    return PartConstraints(strength_target=_strength_label(part.get("type"), params))


def parse_ucf(ucf_path: str) -> List[PartSpec]:
    ucf_path = Path(ucf_path)
    with open(ucf_path) as f:
        ucf_data = json.load(f)

    parts_map = _build_parts_map(ucf_data)
    host      = _infer_host(ucf_data)
    structures = [x for x in ucf_data if x.get("collection") == "structures"]

    all_cassettes = []
    for structure in structures:
        components = _get_cassette_components(structure)
        if components:
            all_cassettes.append((structure, components))

    circuit_total = sum(len(comps) for _, comps in all_cassettes)
    specs, global_pos = [], 0

    for structure, components in all_cassettes:
        gate_name = structure["name"].replace("_structure", "")
        seqs = [
            _clean_seq(parts_map[name]["dnasequence"]) if name in parts_map else ""
            for name in components
        ]
        for i, comp_name in enumerate(components):
            part = parts_map.get(comp_name)
            if part is None:
                global_pos += 1
                continue
            ucf_type = part.get("type", "")
            specs.append(PartSpec(
                part_id=f"{gate_name}__{comp_name}",
                part_type=_UCF_TYPE_TO_PART_TYPE.get(ucf_type, PartType.SPACER),
                host=host,
                functional_role=_UCF_TYPE_TO_ROLE.get(ucf_type, "unknown"),
                upstream_context="".join(seqs[:i]),
                downstream_context="".join(seqs[i + 1:]),
                reference_seq=_clean_seq(part["dnasequence"]),
                constraints=_build_constraints(part),
                circuit_position=global_pos,
                circuit_total=circuit_total,
                sbol_component_id=comp_name,
            ))
            global_pos += 1

    return specs

In [ ]:
# ── LocalEvo2Scorer ───────────────────────────────────────────────────────────
# Scores a fixed continuation sequence given a prompt using the local evo2 model.
# Uses model.score_sequences() which returns total log-likelihood of the full sequence.
# Log-prob is normalized by continuation length (mean per-token, matching ablation study).

class LocalEvo2Scorer:
    def __init__(self, model_id: str = "evo2_1b_base"):
        self.model_id = model_id
        self._model = None
        print(f"LocalEvo2Scorer initialized (model: {self.model_id})")
        print("Model will be loaded on first score() call.")

    def _load(self):
        if self._model is None:
            print(f"Loading {self.model_id} ... (first call, may take a few minutes)")
            from evo2 import Evo2
            self._model = Evo2(self.model_id)
            print("Model loaded.")
        return self._model

    def score(self, prompt: str, continuation: str, downstream: str = "") -> float:
        """Return mean per-token log-prob of `continuation` given `prompt`.

        Scores the full sequence (prompt + continuation + downstream) then
        normalises by len(continuation) to get the per-token score for the RBS.
        Including downstream context mirrors FULL_CONTEXT mode from the ablation.
        """
        model = self._load()
        full_seq = prompt + continuation + downstream
        result = model.score_sequences(sequences=[full_seq])
        if result and len(result) > 0:
            return float(result[0]) / max(len(continuation), 1)
        return 0.0

## Section 4 — Load Kosuri Dataset

In [ ]:
import pandas as pd
import numpy as np
import re

df_raw = pd.read_csv(RBS_CSV)

# Strip surrounding triple-quote artifacts from string columns
for col in df_raw.select_dtypes(include="object").columns:
    df_raw[col] = df_raw[col].str.strip('"').str.strip()

df = df_raw.copy()

# Clean sequences: remove whitespace, uppercase
df["seq_clean"] = df["Sequence"].str.replace(r"\s+", "", regex=True).str.upper()

if STRIP_START_CODON:
    # Remove trailing CATATG (start codon used in the Kosuri CAT reporter; not part of the RBS)
    df["seq_clean"] = df["seq_clean"].str.replace(r"CATATG$", "", regex=True)

# Validate: keep only pure ACGT sequences with length > 0
valid_mask = df["seq_clean"].str.fullmatch(r"[ACGT]+", na=False)
n_dropped = (~valid_mask).sum()
if n_dropped:
    print(f"Dropping {n_dropped} rows with invalid/empty sequences.")
df = df[valid_mask].dropna(subset=["mean.xlat"]).reset_index(drop=True)

print(f"Dataset: {len(df)} RBS sequences")
print(f"mean.xlat range: {df['mean.xlat'].min():.0f} – {df['mean.xlat'].max():.0f}")
print(f"Sequence length range: {df['seq_clean'].str.len().min()} – {df['seq_clean'].str.len().max()} bp")
df[["RBS", "seq_clean", "mean.xlat"]].head(8)

## Section 5 — Parse UCF and Extract Circuit Context

In [ ]:
parts = parse_ucf(UCF_PATH)
print(f"Parsed {len(parts)} parts from UCF.")

# Find the RBS part for A1_AmtR gate (sbol_component_id == 'A1')
rbs_parts = [p for p in parts if p.part_type == PartType.RBS and GATE_NAME in p.part_id]
print(f"\nRBS parts in {GATE_NAME}:")
for p in rbs_parts:
    print(f"  {p.part_id}  sbol={p.sbol_component_id}  ref_seq={p.reference_seq}  ({len(p.reference_seq or '')} bp)")

rbs_part = rbs_parts[0]
upstream   = rbs_part.upstream_context
downstream = rbs_part.downstream_context

print(f"\nUpstream context  ({len(upstream)} bp): ...{upstream[-30:]}")
print(f"Native RBS        ({len(rbs_part.reference_seq or '')} bp): {rbs_part.reference_seq}")
print(f"Downstream context ({len(downstream)} bp): {downstream[:30]}...")

## Section 6 — Score Each RBS with Evo2

In [ ]:
from tqdm.notebook import tqdm

scorer = LocalEvo2Scorer(model_id=MODEL_ID)

log_probs = []
errors    = []

for seq in tqdm(df["seq_clean"], desc="Scoring RBS sequences"):
    try:
        lp = scorer.score(prompt=upstream, continuation=seq, downstream=downstream)
        log_probs.append(lp)
        errors.append(None)
    except Exception as e:
        log_probs.append(float("nan"))
        errors.append(str(e))

df["evo2_log_prob"] = log_probs
df["score_error"]   = errors

n_failed = df["score_error"].notna().sum()
print(f"\nScored {len(df) - n_failed}/{len(df)} sequences successfully.")
if n_failed:
    print("Failed sequences:")
    print(df[df["score_error"].notna()][["RBS", "seq_clean", "score_error"]])

## Section 7 — Results Table

In [ ]:
df_scored = df.dropna(subset=["evo2_log_prob"]).copy()
df_scored["log10_xlat"] = np.log10(df_scored["mean.xlat"])

display_cols = ["RBS", "seq_clean", "mean.xlat", "log10_xlat", "evo2_log_prob"]
print(f"Results for {len(df_scored)} sequences (sorted by evo2_log_prob):")
df_scored[display_cols].sort_values("evo2_log_prob", ascending=False).reset_index(drop=True)

## Section 8 — Statistical Analysis

In [ ]:
from scipy.stats import pearsonr, spearmanr, kendalltau

x  = df_scored["evo2_log_prob"].values
y  = df_scored["mean.xlat"].values
ly = df_scored["log10_xlat"].values

r,    p_r    = pearsonr(x, y)
rho,  p_rho  = spearmanr(x, y)
tau,  p_tau  = kendalltau(x, y)
r_log, p_rlog = pearsonr(x, ly)

alpha = 0.05

stats_rows = [
    {"Test": "Pearson r (raw)",        "Coefficient": r,     "p-value": p_r,    "Significant": p_r    < alpha},
    {"Test": "Pearson r (log10 xlat)", "Coefficient": r_log, "p-value": p_rlog, "Significant": p_rlog < alpha},
    {"Test": "Spearman ρ",             "Coefficient": rho,   "p-value": p_rho,  "Significant": p_rho  < alpha},
    {"Test": "Kendall τ",              "Coefficient": tau,   "p-value": p_tau,  "Significant": p_tau  < alpha},
]

stats_df = pd.DataFrame(stats_rows)
stats_df["Coefficient"] = stats_df["Coefficient"].map(lambda v: f"{v:.4f}")
stats_df["p-value"]     = stats_df["p-value"].map(lambda v: f"{v:.4f}")
stats_df["Significant"] = stats_df["Significant"].map(lambda v: "Yes" if v else "No")

print(f"n = {len(df_scored)} RBS sequences, α = {alpha}")
print()
print(stats_df.to_string(index=False))

## Section 9 — Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f"Evo2 Log-Probability vs. Translation Rate\n"
    f"A1_AmtR circuit context | {MODEL_ID} | n={len(df_scored)}",
    fontsize=13,
)

# ── Left: raw scale ──────────────────────────────────────────────────────────
ax = axes[0]
ax.scatter(x, y, alpha=0.6, edgecolors="none", s=40, color="steelblue")
m_raw, b_raw = np.polyfit(x, y, 1)
xline = np.linspace(x.min(), x.max(), 200)
ax.plot(xline, m_raw * xline + b_raw, color="tomato", linewidth=1.5)

# Annotate extremes
for _, row in df_scored.nlargest(2, "mean.xlat").iterrows():
    ax.annotate(row["RBS"], (row["evo2_log_prob"], row["mean.xlat"]),
                fontsize=7, xytext=(4, 4), textcoords="offset points")
for _, row in df_scored.nsmallest(2, "mean.xlat").iterrows():
    ax.annotate(row["RBS"], (row["evo2_log_prob"], row["mean.xlat"]),
                fontsize=7, xytext=(4, -10), textcoords="offset points")

ax.set_xlabel("Evo2 log-prob (mean per token)")
ax.set_ylabel("mean.xlat (relative translation rate)")
ax.set_title(f"Raw scale  |  r = {float(r):.3f}, p = {float(p_r):.4f}")

# ── Right: log10 scale ───────────────────────────────────────────────────────
ax = axes[1]
ax.scatter(x, ly, alpha=0.6, edgecolors="none", s=40, color="mediumseagreen")
m_log, b_log = np.polyfit(x, ly, 1)
ax.plot(xline, m_log * xline + b_log, color="tomato", linewidth=1.5)

for _, row in df_scored.nlargest(2, "log10_xlat").iterrows():
    ax.annotate(row["RBS"], (row["evo2_log_prob"], row["log10_xlat"]),
                fontsize=7, xytext=(4, 4), textcoords="offset points")
for _, row in df_scored.nsmallest(2, "log10_xlat").iterrows():
    ax.annotate(row["RBS"], (row["evo2_log_prob"], row["log10_xlat"]),
                fontsize=7, xytext=(4, -10), textcoords="offset points")

ax.set_xlabel("Evo2 log-prob (mean per token)")
ax.set_ylabel("log₁₀(mean.xlat)")
ax.set_title(f"Log₁₀ scale  |  r = {float(r_log):.3f}, p = {float(p_rlog):.4f}")

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "rbs_kosuri_correlation.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"Figure saved to {fig_path}")
plt.show()

## Section 10 — Save Results

In [ ]:
# Full scored dataset
results_path = os.path.join(OUTPUT_DIR, "rbs_kosuri_scores.csv")
df_scored[["RBS", "full.name", "seq_clean", "mean.xlat", "log10_xlat", "evo2_log_prob"]].to_csv(
    results_path, index=False
)
print(f"Scores saved to {results_path}")

# Stats summary
stats_raw = [
    {"test": "pearson_r_raw",       "coefficient": float(r),     "p_value": float(p_r)},
    {"test": "pearson_r_log10",     "coefficient": float(r_log), "p_value": float(p_rlog)},
    {"test": "spearman_rho",        "coefficient": float(rho),   "p_value": float(p_rho)},
    {"test": "kendall_tau",         "coefficient": float(tau),   "p_value": float(p_tau)},
]
stats_path = os.path.join(OUTPUT_DIR, "rbs_kosuri_stats.csv")
pd.DataFrame(stats_raw).to_csv(stats_path, index=False)
print(f"Stats saved to {stats_path}")

print(f"\nAll results written to {OUTPUT_DIR}/")